# 00 - Data Audit

FRTBOT research POC — country-level vertical slice (SPEC.md M1/M2).

This notebook downloads a bounded real-data sample for all five configured
market proxies and FX pairs, then runs the SPEC.md-required data audit
(coverage, gaps, staleness, currency, adjustment metadata; usable / proxy /
disabled / missing / stale / synthetic classification).

**Data is REAL** (Yahoo Finance via `yfinance`), not `SYNTHETIC` — see the
`source` column below. Retrieved on the date this notebook was last executed.

In [1]:
from datetime import date
from pathlib import Path

import pandas as pd

from frtbot.config import load_markets_config
from frtbot.data.cache import DataCache
from frtbot.data.providers import get_provider
from frtbot.reporting.data_audit import build_data_audit

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

START = date(2012, 6, 1)
END = date(2026, 7, 31)  # bounded sample; see README for the deferred full-history command

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists():
    REPO_ROOT = REPO_ROOT.parent

markets_config = load_markets_config(REPO_ROOT / "configs" / "markets.example.yml")
cache = DataCache(root=REPO_ROOT / "data")
provider = get_provider("yfinance")

print(f"Base currency: {markets_config.base_currency}")
print(f"Markets: {[m.key for m in markets_config.markets]}")

Base currency: THB
Markets: ['US', 'EU', 'JP', 'CN', 'TH']


In [2]:
for m in markets_config.markets:
    if m.mode == "disabled":
        print(f"{m.key}: disabled ({m.disabled_reason}) - skipping download")
        continue
    df, meta = cache.get_or_fetch(m.key, m.provider_symbol, "ohlcv", provider, START, END)
    print(f"{m.key:4s} {m.provider_symbol:12s} rows={len(df):5d} "
          f"{df.index.min().date()} .. {df.index.max().date()} source={meta['source']}")

from frtbot.data.fx import fetch_fx_series

for fx in markets_config.fx:
    rate, meta = fetch_fx_series(fx, cache, provider, START, END)
    print(f"{fx.pair:8s} {fx.provider_symbol:10s} rows={len(rate):5d} "
          f"{rate.index.min().date()} .. {rate.index.max().date()} "
          f"invert={fx.invert} source={meta['source']}")

US   SPY          rows= 3560 2012-06-01 .. 2026-07-30 source=REAL
EU   EXSA.DE      rows= 3590 2012-06-01 .. 2026-07-30 source=REAL
JP   1306.T       rows= 3483 2012-06-01 .. 2026-07-30 source=REAL
CN   510300.SS    rows= 3438 2012-06-01 .. 2026-07-30 source=REAL
TH   TDEX.BK      rows= 3449 2012-06-01 .. 2026-07-30 source=REAL
USDTHB   USDTHB=X   rows= 3686 2012-06-01 .. 2026-07-30 invert=False source=REAL
EURTHB   EURTHB=X   rows= 3686 2012-06-01 .. 2026-07-30 invert=False source=REAL
JPYTHB   JPYTHB=X   rows= 3687 2012-06-01 .. 2026-07-30 invert=False source=REAL
CNYTHB   THBCNY=X   rows= 3686 2012-06-01 .. 2026-07-30 invert=True source=REAL


## Data audit report

In [3]:
audit = build_data_audit(markets_config, cache, as_of=pd.Timestamp(END))
display_cols = [
    "key", "kind", "status", "source", "currency", "provider", "provider_symbol",
    "adjustment_type", "first_valid_date", "last_valid_date", "row_count",
    "business_day_coverage_ratio", "max_gap_days", "price_anomaly_dates",
]
audit[display_cols]

,key,kind,status,source,currency,provider,provider_symbol,adjustment_type,first_valid_date,last_valid_date,row_count,business_day_coverage_ratio,max_gap_days,price_anomaly_dates
0,US,market,proxy,REAL,USD,yfinance,SPY,split_dividend_adjusted,2012-06-01,2026-07-30,3560,0.963464,5,[]
1,EU,market,proxy,REAL,EUR,yfinance,EXSA.DE,split_dividend_adjusted,2012-06-01,2026-07-30,3590,0.971583,6,[]
2,JP,market,proxy,REAL,JPY,yfinance,1306.T,split_dividend_adjusted,2012-06-01,2026-07-30,3483,0.942625,11,"[2015-01-05, 2026-03-30, 2026-04-01]"
3,CN,market,proxy,REAL,CNY,yfinance,510300.SS,split_dividend_adjusted,2012-06-01,2026-07-30,3438,0.930447,11,[]
4,TH,market,proxy,REAL,THB,yfinance,TDEX.BK,split_dividend_adjusted,2012-06-01,2026-07-30,3449,0.933424,6,[]
5,USDTHB,fx,proxy,REAL,USD,yfinance,USDTHB=X,NaN,2012-06-01,2026-07-30,3686,0.997564,5,[]
6,EURTHB,fx,proxy,REAL,EUR,yfinance,EURTHB=X,NaN,2012-06-01,2026-07-30,3686,0.997564,5,[]
7,JPYTHB,fx,proxy,REAL,JPY,yfinance,JPYTHB=X,NaN,2012-06-01,2026-07-30,3687,0.997835,5,[]
8,CNYTHB,fx,proxy,REAL,CNY,yfinance,THBCNY=X,NaN,2012-06-01,2026-07-30,3686,0.997564,5,[]


In [4]:
print("Status counts:")
print(audit["status"].value_counts())
assert (audit["status"].isin(["proxy", "usable"])).all(), "Unexpected non-usable series in the audit"
assert (audit["source"] == "REAL").all(), "Expected only REAL data in this notebook run"
print("\nAll five markets + four FX pairs are REAL and usable/proxy - no disabled/missing/stale series.")

anomalous = audit[audit["price_anomaly_dates"].apply(len) > 0]
if len(anomalous):
    print("\nWARNING: implausible single-day price move(s) detected (likely an upstream "
          "provider/split defect, not real price action) - SPEC.md section 3 data-quality gate:")
    print(anomalous[["key", "provider_symbol", "price_anomaly_dates"]].to_string(index=False))
    print("\nSee frtbot.data.quality.detect_extreme_return_days. These markets are excluded "
          "from the modeling/backtest notebooks (01, 02) and scripts/run_full_backtest.py for "
          "any affected run - see their 'data-quality gate' cell.")

Status counts:
status
proxy    9
Name: count, dtype: int64

All five markets + four FX pairs are REAL and usable/proxy - no disabled/missing/stale series.

key provider_symbol                  price_anomaly_dates
 JP          1306.T [2015-01-05, 2026-03-30, 2026-04-01]

See frtbot.data.quality.detect_extreme_return_days. These markets are excluded from the modeling/backtest notebooks (01, 02) and scripts/run_full_backtest.py for any affected run - see their 'data-quality gate' cell.
